# Testes com usuários finais

Perfil dos participantes, conclusão e facilidade das tarefas, SUS, UEQ-S e confiança, com comparação entre os grupos por página, dentro de cada usuário e por perfil.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

DADOS = Path("../dados")

# Verifica se as planilhas estão na pasta dados/ (ao lado da pasta notebooks/)
if not DADOS.exists():
    raise FileNotFoundError(f"Pasta de dados não encontrada: {DADOS.resolve()}. Crie a pasta dados/ ao lado de notebooks/ e coloque nela as planilhas.")


## Funções auxiliares

In [2]:
def coluna(df, trecho):
    """Encontra a coluna cujo título contém o trecho informado."""
    achadas = [c for c in df.columns if trecho.lower() in c.lower()]
    if not achadas:
        raise KeyError(f"Coluna não encontrada: {trecho}")
    return df[achadas[0]]


def delta_cliff(a, b):
    """Delta de Cliff: positivo quando b (G2) tende a ser maior que a (G1)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    maior = sum((y > x) for x in a for y in b)
    menor = sum((y < x) for x in a for y in b)
    return (maior - menor) / (len(a) * len(b))


def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)


def comparar(nome, a, b):
    """Mediana, média, intervalo interquartil, Mann-Whitney e delta de Cliff."""
    a, b = pd.Series(a, dtype=float), pd.Series(b, dtype=float)
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    return {"Variável": nome,
            "Md G1": round(a.median(), 2), "IIQ G1": round(iqr(a), 2),
            "Md G2": round(b.median(), 2), "IIQ G2": round(iqr(b), 2),
            "M G1": round(a.mean(), 2), "M G2": round(b.mean(), 2),
            "delta de Cliff": round(delta_cliff(a, b), 2), "p": round(p, 3)}


def holm(p):
    """Correção de Holm para comparações múltiplas."""
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); mx = 0
    for k, i in enumerate(o):
        mx = max(mx, min(1, (m - k) * p[i])); adj[i] = mx
    return adj

In [3]:
perfil = pd.read_excel(DADOS / "Questionario_Usuario_Final_respostas.xlsx")
aval = pd.read_excel(DADOS / "Questionario_avaliacao_landing_page_usuario_final_respostas.xlsx")
perfil.columns = [c.strip() for c in perfil.columns]
aval.columns = [c.strip() for c in aval.columns]

# O código do site é o código do profissional que o construiu
G1 = {1, 3, 4, 6, 7, 9, 12, 18}
G2 = {2, 5, 8, 10, 13, 14, 19, 20}
SEM_ACESS = 3  # única página cujo autor declarou não ter considerado a acessibilidade

assert perfil.iloc[:, 2].is_unique, "Há códigos de participante repetidos no perfil"
assert not aval.duplicated(subset=[aval.columns[1], aval.columns[2]]).any(), "Há páginas repetidas para o mesmo participante"

In [4]:
pf = pd.DataFrame({
    "part": perfil.iloc[:, 2].astype(int), "idade": perfil.iloc[:, 3].astype(int),
    "genero": perfil.iloc[:, 4], "escolaridade": perfil.iloc[:, 5], "dificuldade": perfil.iloc[:, 6],
    "lingua": perfil.iloc[:, 7], "frequencia": perfil.iloc[:, 8], "aparelho": perfil.iloc[:, 9],
    "autonomia": perfil.iloc[:, 11], "barreiras": perfil.iloc[:, 12], "desistiu": perfil.iloc[:, 13],
    "recursos": perfil.iloc[:, 14],
})

# Pessoa com deficiência (critério funcional do IBGE) ou idosa (60+).
# 1009 marcou todas as dificuldades e também "Não tenho"; é classificado só pela idade.
PERFIS = {1003: "Deficiência visual (daltonismo)", 1004: "Deficiência visual (baixa visão)",
          1005: "Deficiência visual (cegueira)", 1006: "Deficiência auditiva (surdez)",
          1007: "Deficiência física (criança)", 1016: "Limitação nas funções mentais",
          1008: "Pessoa idosa", 1009: "Pessoa idosa"}
pf["perfil"] = pf.part.map(PERFIS).fillna("Sem deficiência")
pf["pcd_idoso"] = pf.part.isin(PERFIS)

def num(s):
    return pd.to_numeric(s.astype(str).str.extract(r"(\d)")[0], errors="coerce")

av = pd.DataFrame({"part": aval.iloc[:, 1].astype(int), "site": aval.iloc[:, 2].astype(int),
                   "aparelho": aval.iloc[:, 3], "recurso": aval.iloc[:, 4],
                   "erro_entendeu": aval.iloc[:, 15], "confia": aval.iloc[:, 37]})
av["grupo"] = np.where(av.site.isin(G1), "G1", "G2")
for t in range(5):
    av[f"t{t+1}"] = aval.iloc[:, 5 + 2 * t]
    av[f"t{t+1}_ok"] = (aval.iloc[:, 5 + 2 * t] == "Sim, sozinho(a)").astype(int)
    av[f"t{t+1}_fac"] = aval.iloc[:, 6 + 2 * t].astype(float)
ok = [f"t{t}_ok" for t in range(1, 6)]
fac = [f"t{t}_fac" for t in range(1, 6)]
av["conclusao"] = av[ok].mean(axis=1) * 100
av["facilidade"] = av[fac].mean(axis=1)

sus = aval.iloc[:, 16:26].apply(num)
av["sus"] = ((sus.iloc[:, 0::2] - 1).sum(axis=1) + (5 - sus.iloc[:, 1::2]).sum(axis=1)) * 2.5

ueq = aval.iloc[:, 26:34].astype(float) - 4
av["ueq_prag"] = ueq.iloc[:, :4].mean(axis=1)
av["ueq_hed"] = ueq.iloc[:, 4:].mean(axis=1)
av["ueq_geral"] = ueq.mean(axis=1)
av["confia_sim"] = (av.confia == "Sim").astype(int)
av = av.merge(pf[["part", "perfil", "pcd_idoso"]], on="part", how="left")

def faixa_sus(x):
    for lim, nome in [(20.5, "pior imaginável"), (39.9, "ruim"), (52.5, "aceitável"),
                      (73.9, "boa"), (85.9, "excelente"), (100, "melhor imaginável")]:
        if x <= lim:
            return nome
av["faixa_sus"] = av.sus.apply(faixa_sus)
print(f"Participantes: {pf.part.nunique()} | avaliações: {len(av)} | concluíram as 3: {(av.part.value_counts() == 3).sum()}")

Participantes: 17 | avaliações: 49 | concluíram as 3: 16


### 2.1 Perfil dos participantes

In [5]:
print("Idade: mediana", pf.idade.median(), "| mín.", pf.idade.min(), "| máx.", pf.idade.max())
for c in ["genero", "escolaridade", "aparelho", "frequencia", "autonomia", "desistiu", "lingua", "perfil"]:
    print(f"\n{c}:\n{pf[c].value_counts().to_string()}")
print("\nAparelho no teste:\n", av.aparelho.value_counts().to_string())
print("\nAvaliações por grupo:\n", av.groupby("grupo").size().to_string())
print("\nAvaliações de pessoas com deficiência ou idosas:\n", av[av.pcd_idoso].groupby("grupo").size().to_string())

Idade: mediana 32.0 | mín. 7 | máx. 78

genero:
genero
Mulher    9
Homem     8

escolaridade:
escolaridade
Ensino médio completo            5
Ensino superior completo         4
Ensino técnico                   3
Pós-graduação                    2
Ensino fundamental completo      2
Ensino fundamental incompleto    1

aparelho:
aparelho
Celular                   13
Computador ou notebook     3
Tablet                     1

frequencia:
frequencia
Todos os dias               16
Algumas vezes por semana     1

autonomia:
autonomia
Consigo usar sozinho(a)                        12
Consigo sozinho(a), mas às vezes peço ajuda     4
Quase sempre preciso de ajuda                   1

desistiu:
desistiu
Sim, poucas vezes    8
Sim, várias vezes    5
Nunca                4

lingua:
lingua
Português                        16
Português e Libras igualmente     1

perfil:
perfil
Sem deficiência                     9
Pessoa idosa                        2
Deficiência visual (cegueira)       1
Deficiência

### 2.2 Tarefas (Tabela 4)

In [6]:
tab = av.groupby("grupo")[ok + fac].mean()
tab[ok] *= 100
tab.round(2).T

grupo,G1,G2
t1_ok,96.15,95.65
t2_ok,96.15,95.65
t3_ok,92.31,95.65
t4_ok,100.00,95.65
t5_ok,84.62,78.26
t1_fac,6.58,6.61
t2_fac,6.31,6.48
t3_fac,6.35,6.30
t4_fac,6.65,6.48
t5_fac,6.12,6.04


### 2.3 SUS e UEQ-S por avaliação

In [7]:
print(av.groupby("grupo")[["sus", "ueq_prag", "ueq_hed", "ueq_geral"]].agg(["mean", "median"]).round(2).T.to_string())
print("\nTotal:", av[["sus", "ueq_prag", "ueq_hed", "ueq_geral"]].mean().round(2).to_dict())
print("\nFaixas da SUS:\n", av.groupby("grupo").faixa_sus.value_counts().unstack(0).fillna(0).astype(int).to_string())
print("\nConfiança:\n", av.groupby("grupo").confia.value_counts().unstack(0).fillna(0).astype(int).to_string())

grupo                G1     G2
sus       mean    79.33  80.87
          median  76.25  77.50
ueq_prag  mean     2.02   1.99
          median   2.38   2.00
ueq_hed   mean     1.18   1.30
          median   1.50   1.50
ueq_geral mean     1.60   1.65
          median   1.69   1.88

Total: {'sus': 80.05, 'ueq_prag': 2.01, 'ueq_hed': 1.24, 'ueq_geral': 1.62}

Faixas da SUS:
 grupo              G1  G2
faixa_sus                
aceitável           1   0
boa                 9   6
excelente           7   5
melhor imaginável   9  11
ruim                0   1

Confiança:
 grupo   G1  G2
confia        
Não      2   0
Sim     15  12
Talvez   6   9


### 2.4 Comparação por página (Tabela 5)

In [8]:
metricas = {"Conclusão sem ajuda (%)": "conclusao", "Facilidade média (1 a 7)": "facilidade",
            "SUS (0 a 100)": "sus", "UEQ-S pragmática": "ueq_prag", "UEQ-S hedônica": "ueq_hed",
            "UEQ-S geral": "ueq_geral", "Confiaria (proporção Sim)": "confia_sim"}
pag = av.groupby(["site", "grupo"])[list(metricas.values())].mean().reset_index()
comp = pd.DataFrame([comparar(n, pag.loc[pag.grupo == "G1", c], pag.loc[pag.grupo == "G2", c])
                     for n, c in metricas.items()])
comp["p Holm"] = np.round(holm(comp.p.values), 3)
comp

,Variável,Md G1,IIQ G1,Md G2,IIQ G2,M G1,M G2,delta de Cliff,p,p Holm
0,Conclusão sem ajuda (%),100.00,8.33,93.33,10.00,95.71,92.50,-0.25,0.396,1.0
1,Facilidade média (1 a 7),6.43,0.30,6.47,0.70,6.45,6.41,0.00,1.000,1.0
2,SUS (0 a 100),80.83,11.88,85.83,10.62,80.31,81.04,0.08,0.833,1.0
3,UEQ-S pragmática,1.96,0.40,2.12,0.33,2.08,2.01,0.08,0.833,1.0
4,UEQ-S hedônica,1.10,0.91,1.25,0.73,1.16,1.30,0.20,0.526,1.0
5,UEQ-S geral,1.71,0.51,1.71,0.32,1.62,1.66,0.08,0.833,1.0
6,Confiaria (proporção Sim),0.67,0.60,0.58,0.33,0.64,0.52,-0.22,0.483,1.0


In [9]:
# Média de cada página
pag.sort_values(["grupo", "sus"]).round(2)

,site,grupo,conclusao,facilidade,sus,ueq_prag,ueq_hed,ueq_geral,confia_sim
13,18,G1,93.33,6.00,66.67,1.42,0.17,0.79,0.00
2,3,G1,85.71,6.20,75.00,1.86,1.11,1.48,0.43
8,9,G1,100.00,6.53,75.83,2.25,1.75,2.00,1.00
3,4,G1,100.00,6.60,77.50,1.88,0.75,1.31,1.00
5,6,G1,100.00,6.33,84.17,1.92,1.83,1.88,0.33
0,1,G1,100.00,7.00,87.50,2.00,1.08,1.54,0.67
6,7,G1,100.00,6.60,87.50,3.00,0.88,1.94,1.00
10,12,G1,86.67,6.33,88.33,2.33,1.75,2.04,0.67
11,13,G2,93.33,6.13,67.50,1.00,0.50,0.75,0.00
15,20,G2,80.00,5.47,67.50,1.75,1.83,1.79,0.67


### 2.5 Comparação dentro de cada usuário (Wilcoxon)

In [10]:
pu = av.groupby(["part", "grupo"])[["sus", "ueq_geral", "facilidade"]].mean().unstack().dropna()
for c in ["sus", "ueq_geral", "facilidade"]:
    dif = pu[(c, "G2")] - pu[(c, "G1")]
    w = stats.wilcoxon(pu[(c, "G1")], pu[(c, "G2")]) if (dif != 0).any() else None
    print(f"{c}: n = {len(pu)}, mediana da diferença G2 - G1 = {dif.median():.2f}"
          + (f", Wilcoxon p = {w.pvalue:.3f}" if w else ""))

sus: n = 16, mediana da diferença G2 - G1 = 0.00, Wilcoxon p = 0.730
ueq_geral: n = 16, mediana da diferença G2 - G1 = 0.16, Wilcoxon p = 0.362
facilidade: n = 16, mediana da diferença G2 - G1 = 0.00, Wilcoxon p = 0.574


### 2.6 Resultados por perfil e página 003

In [11]:
print(av.groupby(["pcd_idoso", "grupo"])[["conclusao", "facilidade", "sus", "ueq_geral", "confia_sim"]].mean().round(2).to_string())

q = av[av.part.isin(av.loc[av.site == SEM_ACESS, "part"])].copy()
q["e003"] = q.site == SEM_ACESS
print("\nPágina 003 x demais páginas dos mesmos usuários:")
print(q.groupby("e003")[["sus", "ueq_geral", "facilidade", "conclusao", "confia_sim"]].mean().round(2).to_string())

                 conclusao  facilidade    sus  ueq_geral  confia_sim
pcd_idoso grupo                                                     
False     G1         98.18        6.53  82.50       1.43        0.64
          G2         97.50        6.64  81.88       1.48        0.44
True      G1         90.67        6.31  77.00       1.72        0.53
          G2         80.00        5.80  78.57       2.04        0.71

Página 003 x demais páginas dos mesmos usuários:
         sus  ueq_geral  facilidade  conclusao  confia_sim
e003                                                      
False  81.25       2.28        6.17      85.00        0.67
True   75.00       1.48        6.20      85.71        0.43


### 2.7 Preferência declarada

In [12]:
pref = aval.iloc[:, [1, 39]].dropna()
pref.columns = ["part", "resposta"]
pref

,part,resposta
2,1000,019
6,1004,004
9,1001,014
14,1002,o site do codigo 013
18,1016,009
20,1015,001
21,1003,001
22,1010,Vi só o primeiro.
23,1010,Não tenho.
24,1010,012
